# Identifier une molécule à partir de ses spectres de masse (CASMI 2026)

## 1. Problème et démarche

Pour chaque molécule inconnue, on dispose d'un ou plusieurs spectres MS/MS et on propose jusqu'à 25 SMILES classés. Le score est le MRR@25 : la moyenne de 1/rang de la bonne structure, 0 si elle n'est pas dans les 25. Une structure est juste si les 14 premiers caractères de son InChIKey sont les bons ; la stéréochimie ne compte pas.

Un classifieur à une classe par molécule ne reconnaîtrait que des molécules déjà vues. On prédit donc un fingerprint de Morgan, puis on cherche les structures proches dans un catalogue :

1. masse neutre, à partir de `precursor_mz` et de l'`adduct` ;
2. candidats du catalogue à ±10 ppm ;
3. fingerprint prédit à partir du spectre (seule étape apprise) ;
4. tri des candidats par similarité de Tanimoto, et top 25.

La validation porte sur des molécules absentes de l'entraînement. Le test visible n'y sert pas, car ses spectres sont des copies de spectres du train.

### Imports et chemins

Le notebook ne dépend d'aucun autre fichier du projet. Sur Kaggle, il lit les données attachées dans `/kaggle/input/`. En local, il les lit dans `data/raw/` et les télécharge avec `kagglehub` si elles n'y sont pas encore ; il faut pour cela avoir accepté les règles de la compétition et enregistré un jeton d'API dans `~/.kaggle/access_token`. Les fichiers produits vont dans `data/rapport/`. Paquets nécessaires : pandas, pyarrow, numpy, matplotlib, scikit-learn, rdkit et kagglehub (torch pour le MLP). La cellule doit afficher les trois fichiers.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

SEED = 42
COMPETITION = "enveda-CASMI26-molecule-id-mass-spectra"

KAGGLE_INPUT = Path("/kaggle/input")
if KAGGLE_INPUT.exists():  # sur Kaggle : données attachées au notebook, pas d'Internet
    RAW_DIR = next(KAGGLE_INPUT.rglob("train.parquet")).parent
    WORK_DIR = Path("/kaggle/working")
else:
    RAW_DIR = Path("data/raw")
    WORK_DIR = Path("data/rapport")
    if not (RAW_DIR / "train.parquet").exists():
        # Téléchargement (environ 3 Go) : il faut avoir accepté les règles de la compétition
        # et enregistré un jeton d'API Kaggle dans ~/.kaggle/access_token
        import kagglehub
        kagglehub.competition_download(COMPETITION, output_dir=str(RAW_DIR))
WORK_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = RAW_DIR / "train.parquet"
TEST_PATH = RAW_DIR / "test.parquet"
SUBMISSION_PATH = RAW_DIR / "sample_submission.csv"

for path in [TRAIN_PATH, TEST_PATH, SUBMISSION_PATH]:
    print(f"{path}  {path.stat().st_size / 1e6:,.1f} Mo")

## 2. Données

On charge seulement les colonnes utiles du train, sans les pics, qui sont les plus lourds. L'index de `train` est la position du spectre dans le fichier : il servira à relire les pics d'un échantillon. À regarder : le nombre de spectres et de molécules.

Constat : le train compte 2 539 608 spectres pour 275 810 molécules et tient en 0,7 Go sans les pics. Le test visible compte 1 213 spectres pour 400 molécules.

In [ ]:
COLUMNS = ["inchikey14", "normalized_smiles", "ingest_lib", "adduct", "ionization_mode",
           "instrument_type", "precursor_mz", "collision_energy_ev", "num_peaks"]

train = pd.read_parquet(TRAIN_PATH, columns=COLUMNS)
test = pd.read_parquet(TEST_PATH)

print(f"train : {len(train):,} spectres, {train['inchikey14'].nunique():,} molécules, "
      f"{train.memory_usage(deep=True).sum() / 1e9:.1f} Go en mémoire")
print(f"test  : {len(test):,} spectres, {test['molecule_id'].nunique():,} molécules")
display(train.head())
test.head()

### Un spectre et sa molécule

Pour voir à quoi ressemblent les données, on lit les pics des 2 000 premiers spectres seulement, et on en affiche un : une molécule de taille courante (precursor_mz entre 250 et 500) avec une quarantaine de pics. RDKit dessine la molécule à partir de son SMILES. À regarder : les fragments sont tous plus légers que le précurseur, en pointillés.

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw

# Les 2 000 premiers spectres du fichier suffisent pour un exemple : on ne lit pas tous les pics
pf = pq.ParquetFile(TRAIN_PATH)
first = next(pf.iter_batches(batch_size=2000, columns=COLUMNS + ["ms2_mzs", "ms2_normalized_intensities"])).to_pandas()
typical = first[first["precursor_mz"].between(250, 500)]  # une molécule de taille courante
spectrum = typical.iloc[(typical["num_peaks"] - 40).abs().argmin()]  # environ 40 pics, lisible
mzs, intensities = np.asarray(spectrum["ms2_mzs"]), np.asarray(spectrum["ms2_normalized_intensities"])

fig, (ax, ax_mol) = plt.subplots(1, 2, figsize=(13, 4), gridspec_kw={"width_ratios": [2.2, 1]})
ax.vlines(mzs, 0, intensities, color="#2a78d6", linewidth=1.5)
ax.axvline(spectrum["precursor_mz"], color="#9a9893", linestyle="--", linewidth=1)
ax.text(spectrum["precursor_mz"], 1.02, " précurseur", color="#52514e", fontsize=9)
ax.set_xlabel("m/z")
ax.set_ylabel("intensité relative")
ax.set_title(f"Spectre MS/MS : {spectrum['num_peaks']} pics, adduct {spectrum['adduct']}, "
             f"precursor_mz = {spectrum['precursor_mz']:.4f}", loc="left")
ax.spines[["top", "right"]].set_visible(False)

ax_mol.imshow(Draw.MolToImage(Chem.MolFromSmiles(spectrum["normalized_smiles"]), size=(400, 300)))
ax_mol.set_title(f"Molécule : {spectrum['inchikey14']}", loc="left")
ax_mol.axis("off")
plt.tight_layout()
plt.show()
print("SMILES :", spectrum["normalized_smiles"])

### Provenance des spectres

Le train réunit plusieurs bibliothèques de spectres (`ingest_lib`), mesurées dans des laboratoires et sur des appareils différents. Le graphique compte les spectres de chacune, et le tableau ajoute leur part et leur nombre de molécules. À regarder : le poids de la première bibliothèque par rapport aux autres.

Constat : enveda-180 fournit 45,4 % des spectres et 182 941 molécules. Les quatre plus grosses bibliothèques réunissent 88,6 % des spectres.

In [ ]:
counts = train["ingest_lib"].value_counts().sort_values()

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.barh(counts.index, counts.values / 1e3, color="#2a78d6", height=0.7)
ax.set_xlabel("spectres (milliers)")
ax.set_title("Spectres du train par bibliothèque", loc="left")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

molecules = train.groupby("ingest_lib")["inchikey14"].nunique()
pd.DataFrame({"spectres": counts, "% des spectres": (counts / len(train) * 100).round(1),
              "molécules": molecules}).sort_values("spectres", ascending=False)

### Valeurs manquantes

Dans `instrument_type`, certains spectres portent un texte (`"nan"`, `"N/A"` ou une chaîne vide) à la place d'une valeur manquante, et `isna()` ne les voit pas. On les convertit d'abord. Le tableau donne ensuite la part de valeurs manquantes des colonnes que nous utilisons, dans le train et dans le test.

Constat : 1 645 faux manquants convertis. Dans le train, `instrument_type` manque pour 1,37 % des spectres et `collision_energy_ev` pour 13,29 %. Le test visible n'a aucune valeur manquante.

In [ ]:
FAKE_NA = ["nan", "N/A", ""]  # textes mis à la place d'une valeur manquante
n_fake = train["instrument_type"].isin(FAKE_NA).sum()
train["instrument_type"] = train["instrument_type"].mask(train["instrument_type"].isin(FAKE_NA))
print(f"Faux manquants convertis dans instrument_type : {n_fake:,}")

cols = ["adduct", "ionization_mode", "precursor_mz", "instrument_type", "collision_energy_ev"]
display(pd.DataFrame({"train (%)": train[cols].isna().mean() * 100,
                      "test (%)": test[cols].isna().mean() * 100}).round(2))

La carte ci-dessous couvre les 18 colonnes du train. Les colonnes hors pics sont lues une par une ; pour les deux colonnes de pics, trop lourdes, on lit le nombre de valeurs nulles dans les métadonnées du fichier. Chaque ligne est une bibliothèque, puis viennent le train entier et le test visible ; « absente » signale une colonne que le test n'a pas. À regarder : si les manques sont répartis partout ou concentrés dans quelques bibliothèques, et ce qu'il en est dans le test.

Constat : cinq colonnes ont des valeurs manquantes, et tout dépend de la bibliothèque. enveda-180 et enveda-np-examples sont complètes, alors que sept bibliothèques ne renseignent jamais `base_peak_intensity`. Les pics n'ont aucune valeur nulle.

In [ ]:
pf = pq.ParquetFile(TRAIN_PATH)
peak_cols = ["ms2_mzs", "ms2_normalized_intensities"]
other_cols = [c for c in pf.schema_arrow.names if c not in peak_cols]

# Colonnes hors pics : lues une par une, pour ne jamais tout charger en mémoire
missing = pd.DataFrame({c: (train[c] if c in train else pd.read_parquet(TRAIN_PATH, columns=[c])[c]).isna()
                        for c in other_cols})
# Pics : trop lourds à lire, on compte les valeurs nulles dans les métadonnées du fichier
meta = pf.metadata
for c in peak_cols:
    i = [meta.schema.column(k).path.split(".")[0] for k in range(meta.num_columns)].index(c)
    print(f"{c} : {sum(meta.row_group(g).column(i).statistics.null_count for g in range(meta.num_row_groups))} valeur nulle")

order = train["ingest_lib"].value_counts().index
table = (missing.groupby(train["ingest_lib"]).mean() * 100).loc[order]
table.loc["train entier"] = missing.mean() * 100
table.loc["test visible"] = test.isna().mean() * 100  # NaN si la colonne n'existe pas dans le test

fig, ax = plt.subplots(figsize=(11, 5))
ax.imshow(table.to_numpy(dtype=float), cmap="Blues", vmin=0, vmax=100, aspect="auto")
for (i, j), v in np.ndenumerate(table.to_numpy(dtype=float)):
    label = "absente" if np.isnan(v) else (f"{v:.0f}" if v >= 0.5 else ("<1" if v > 0 else ""))
    ax.text(j, i, label, ha="center", va="center", fontsize=8, color="white" if v > 50 else "#52514e")
ax.set_xticks(range(table.shape[1]), table.columns, rotation=40, ha="right")
ax.set_yticks(range(table.shape[0]), table.index)
ax.axhline(len(order) - 0.5, color="#52514e", linewidth=1)
ax.set_title("Valeurs manquantes (% des spectres) par bibliothèque et par colonne", loc="left")
plt.tight_layout()
plt.show()

### Peut-on imputer les valeurs manquantes ?

On teste deux colonnes : `base_peak_intensity` (en log10, car elle varie sur plusieurs ordres de grandeur) et `collision_energy_ev` (la moyenne de la liste d'énergies). Les essais portent sur 500 000 spectres tirés au hasard, pour rester rapides. Les entrées des imputeurs n'utilisent que des colonnes présentes dans le test ; `instrument_type` est regroupé en familles d'appareils, car ses libellés sont très hétérogènes. À regarder : la répartition des familles.

Constat : timsTOF (226 717 spectres), Orbitrap/FT (187 309) et QTOF/TOF (67 209) dominent l'échantillon. 6 870 spectres n'ont pas d'instrument (« Inconnu »).

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

# Familles d'instruments : la première dont le motif apparaît dans le libellé l'emporte
INSTRUMENT_FAMILIES = [
    ("timsTOF", r"tims"),
    ("Orbitrap/FT", r"orbitrap|exactive|qehf|ft\b|qft|itft|hybrid ft"),
    ("QTOF/TOF", r"tof|maxis|impact|synapt"),
    ("Trappe ionique", r"ion trap|iontrap|\blit\b|qit|ltq|-it\b"),
    ("Quadripôle", r"qqq|qq\b|quattro|-q\b"),
]

def instrument_family(s):
    """Famille de chaque libellé d'instrument ; "Inconnu" s'il manque, "Autre" s'il n'est d'aucune famille."""
    lower = s.str.lower()
    family = pd.Series("Autre", index=s.index).mask(s.isna(), "Inconnu")
    for name, pattern in reversed(INSTRUMENT_FAMILIES):
        family = family.mask(lower.str.contains(pattern, regex=True, na=False), name)
    return family

sample = train.sample(500_000, random_state=SEED)
top_adducts = train["adduct"].value_counts().index[:15]
X = pd.DataFrame({
    "precursor_mz": sample["precursor_mz"],
    "log_num_peaks": np.log10(sample["num_peaks"]),
    "mode": sample["ionization_mode"].astype("category"),
    "adduct": sample["adduct"].where(sample["adduct"].isin(top_adducts), "autre").astype("category"),
    "instrument": instrument_family(sample["instrument_type"]).astype("category"),
})
bpi = pd.read_parquet(TRAIN_PATH, columns=["base_peak_intensity"])["base_peak_intensity"]
y_bpi = np.log10(bpi.loc[sample.index])
y_ce = sample["collision_energy_ev"].dropna().map(np.mean).reindex(sample.index)  # moyenne de la liste d'énergies
X["instrument"].value_counts()

On masque des valeurs **observées**, on les prédit, et on mesure l'erreur. Deux masquages :
- au hasard, 20 % des valeurs ;
- une bibliothèque entière à la fois, ce qui reproduit les vrais manques, où une bibliothèque ne renseigne jamais la colonne.

Trois imputeurs, du plus simple au plus riche : la médiane globale, la médiane par famille d'instrument et mode, et un gradient boosting. La cellule prend une dizaine de secondes. À regarder : si l'erreur reste acceptable quand on cache une bibliothèque entière.

Constat :
- pour `base_peak_intensity`, le gradient boosting se trompe d'un facteur 10^0,68 ≈ 5 quand on masque au hasard. Quand on cache gnps, l'erreur monte à 1,95 en log10, soit un facteur 90 environ, et aucun imputeur ne fait mieux que la médiane globale (1,93) ;
- pour `collision_energy_ev`, la médiane globale est la plus mauvaise sur tous les masquages. La médiane par instrument et mode est la meilleure sur enveda-180 (8,99 eV) et pluskal_ms2 (12,70 eV), et le gradient boosting gagne sur les autres, avec des écarts de 1 à 4 eV.

In [ ]:
def imputation_trial(X, y, libs):
    """Erreur absolue moyenne de trois imputeurs sur des valeurs observées que l'on masque."""
    observed = y.notna()
    masks = {"au hasard (20 %)": observed & (np.random.default_rng(SEED).random(len(y)) < 0.2)}
    for lib, n in libs[observed].value_counts().items():
        if n >= 1000:
            masks[f"bibliothèque {lib}"] = observed & (libs == lib)
    rows = []
    for name, masked in masks.items():
        fit = observed & ~masked  # les imputeurs n'apprennent que sur les valeurs restées visibles
        median = y[fit].median()
        by_group = y[fit].groupby([X.loc[fit, "instrument"], X.loc[fit, "mode"]], observed=True).median()
        group_pred = X[masked].join(by_group.rename("pred"), on=["instrument", "mode"])["pred"].fillna(median)
        model = HistGradientBoostingRegressor(categorical_features="from_dtype", random_state=SEED)
        model_pred = model.fit(X[fit], y[fit]).predict(X[masked])
        rows.append({"masquage": name, "valeurs masquées": masked.sum(),
                     "médiane globale": (y[masked] - median).abs().mean(),
                     "médiane par instrument et mode": (y[masked] - group_pred).abs().mean(),
                     "gradient boosting": (y[masked] - model_pred).abs().mean()})
    return pd.DataFrame(rows).set_index("masquage").round(2)

libs = sample["ingest_lib"]
print("base_peak_intensity : erreur absolue moyenne en log10 (1 = un facteur 10)")
display(imputation_trial(X, y_bpi, libs))
print("collision_energy_ev : erreur absolue moyenne en eV")
display(imputation_trial(X, y_ce, libs))

### Une formule pour une partie des énergies manquantes

Certaines bibliothèques donnent l'énergie de collision en NCE (énergie normalisée, en %) dans `collision_energy_orig`. Elle se convertit en eV par la formule **eV = NCE × precursor_mz / 500**, arrondie au dixième. La cellule vérifie la formule sur les spectres qui ont les deux valeurs, puis l'applique aux énergies manquantes. Elle crée la colonne `ce_mean`, la moyenne de la liste d'énergies de chaque spectre. À regarder : la part de formules exactes, et combien d'énergies manquantes elle retrouve.

In [ ]:
raw_ce = pd.read_parquet(TRAIN_PATH, columns=["collision_energy_orig", "collision_energy_orig_units"])

# Énergie NCE simple ("35" ou "35%") ; les plages ("30->50%") et les listes ne sont pas converties
text = raw_ce["collision_energy_orig"].str.strip().str.rstrip("%")
nce = pd.to_numeric(text.where(text.str.fullmatch(r"\d+(\.\d+)?", na=False)), errors="coerce")
nce = nce.where(raw_ce["collision_energy_orig_units"] == "NCE")
ev_formula = (nce * train["precursor_mz"] / 500).round(1)

# Une seule énergie par spectre : la moyenne de la liste
train["ce_mean"] = train["collision_energy_ev"].map(lambda v: np.mean(v) if v is not None and len(v) else np.nan)
single = train["collision_energy_ev"].map(lambda v: v is not None and len(v) == 1)

check = ev_formula.notna() & single
exact = (ev_formula[check] - train.loc[check, "ce_mean"]).abs() <= 0.051
print(f"Spectres qui ont l'énergie en NCE et en eV : {check.sum():,}, formule exacte pour {exact.mean():.2%}")

fill = train["ce_mean"].isna() & ev_formula.notna()
train.loc[fill, "ce_mean"] = ev_formula[fill]
print(f"Énergies manquantes : {fill.sum() + train['ce_mean'].isna().sum():,}, "
      f"retrouvées par la formule : {fill.sum():,}, restantes : {train['ce_mean'].isna().sum():,}")

### Décision pour chaque colonne incomplète

On garde tous les spectres. Retirer ceux qui ont une valeur manquante supprimerait des bibliothèques entières, comme gnps, qui n'a aucune énergie de collision.

| Colonne | Manquants (train) | Décision | Raison |
|---|---|---|---|
| `base_peak_intensity` | 46 % | colonne écartée | Sept bibliothèques ne la renseignent jamais, donc son absence révèle surtout la bibliothèque. Les essais ci-dessus échouent quand on masque une bibliothèque entière. |
| `collision_energy_ev` | 13 % | formule NCE → eV quand le texte d'origine le permet, puis médiane par famille d'instrument et mode, avec une variable indicatrice `ce_manquante` | La formule est exacte, mais ne s'applique qu'à une petite partie des manques. La médiane par groupe fait mieux que la médiane globale sur tous les masquages et reste proche du gradient boosting, pour un code bien plus simple. L'indicatrice vaut 1 quand l'énergie vient de la médiane. |
| `collision_energy_orig` | 13 % | sert seulement à la formule NCE → eV | C'est le texte d'origine de l'énergie, qui n'est pas une feature en soi. |
| `instrument_type` | 1 % | catégorie « Inconnu » | Les manques sont rares, et une catégorie à part évite de deviner l'appareil. |
| `precursor_error_ppm` | moins de 1 % | colonne écartée | Elle est calculée à partir de la vraie structure et n'existe pas dans le test : l'utiliser serait une fuite. |

Les médianes seront calculées sur les spectres d'entraînement seulement, après le split, puis appliquée à la validation et au test. Ces choix servent surtout à l'entraînement, puisque le test visible n'a aucune valeur manquante.

## 3. Nettoyage et split

Certaines étiquettes du train sont fausses : un adduct illisible, un adduct de charge opposée au mode d'ionisation, ou une masse qui ne correspond pas à la structure annoncée. On ne supprime aucune ligne. On calcule deux filtres :
- `keep` : le spectre est fiable pour l'**entraînement**, toutes les règles comprises ;
- `query_ok` : le spectre peut servir de **requête de validation**. Seules comptent les règles qui n'utilisent pas la vraie structure, sinon la validation serait plus facile que le test, où la structure est inconnue.

### Lire les adducts

Un adduct s'écrit `[nM ± groupes]z±` : n molécules M, des groupes ajoutés ou perdus, puis la charge. La cellule le décompose en (n, masse ajoutée ou perdue, charge signée), avec les masses monoisotopiques de RDKit. Le même découpage servira au calcul de la masse neutre. À regarder : les adducts que le parser ne sait pas lire.

In [ ]:
import re

from rdkit import Chem
from rdkit.Chem import Descriptors

PERIODIC_TABLE = Chem.GetPeriodicTable()
ELECTRON_MASS = 0.000548579909  # Da
ADDUCT_RE = re.compile(r"\[(\d*)M((?:[+-]\d*[A-Z][A-Za-z0-9]*)*)\](\d*)([+-])")

def formula_mass(formula):
    """Masse monoisotopique d'une formule brute ("H2O") ; erreur si elle n'est pas lisible."""
    if not re.fullmatch(r"(?:[A-Z][a-z]?\d*)+", formula):
        raise ValueError(formula)
    return sum(PERIODIC_TABLE.GetMostCommonIsotopeMass(el) * int(n or 1)
               for el, n in re.findall(r"([A-Z][a-z]?)(\d*)", formula))

def parse_adduct(adduct):
    """(n molécules M, masse ajoutée ou perdue en Da, charge signée), ou None si l'adduct n'est pas lisible."""
    match = ADDUCT_RE.fullmatch(adduct)
    if match is None:
        return None
    n_mol, terms, charge, sign = match.groups()
    try:
        delta = sum((1 if s == "+" else -1) * int(k or 1) * formula_mass(f)
                    for s, k, f in re.findall(r"([+-])(\d*)([A-Z][A-Za-z0-9]*)", terms))
    except (ValueError, RuntimeError):
        return None
    return int(n_mol or 1), delta, int(charge or 1) * (1 if sign == "+" else -1)

adducts = pd.DataFrame([(a, parse_adduct(a)) for a in train["adduct"].unique()], columns=["adduct", "parsed"])
print("Adducts non lisibles :", adducts.loc[adducts["parsed"].isna(), "adduct"].tolist())

### Deux règles sans la structure

- **Charge contre mode** : un ion négatif (`[M-H]-`) ne peut pas être mesuré en mode positif, ni l'inverse. L'adduct ou le mode est donc mal étiqueté.
- **Adduct douteux** : les précurseurs `[M-CH3]-` sont dispersés et ne correspondent à aucune formule, d'après l'analyse de la masse neutre faite plus tôt dans le projet.

À regarder : les adducts concernés, et le nombre de spectres repérés par chaque règle.

In [ ]:
params = {a: p for a, p in zip(adducts["adduct"], adducts["parsed"]) if p is not None}
charge = train["adduct"].map(lambda a: params[a][2] if a in params else np.nan)
expected = train["ionization_mode"].map({"positive": 1, "negative": -1})

rules = pd.DataFrame({
    "adduct_inconnu": charge.isna(),
    "charge_contre_mode": charge.notna() & (np.sign(charge) != expected),
    "adduct_douteux": train["adduct"] == "[M-CH3]-",
})
display(pd.crosstab(train.loc[rules["charge_contre_mode"], "adduct"],
                    train.loc[rules["charge_contre_mode"], "ionization_mode"]).head(10))
rules.sum().rename("spectres").to_frame()

### L'erreur de masse

On compare la masse neutre déduite du précurseur et de l'adduct à la masse exacte de la vraie structure (RDKit). Un grand écart signale une étiquette fausse : mauvais adduct ou mauvaise structure. Cette règle utilise la structure, donc elle ne filtre que l'entraînement.

On recalcule cet écart au lieu de lire `precursor_error_ppm` : le premier tableau compare les médianes par adduct, et la colonne fournie paraît décalée pour certains adducts. L'histogramme montre la distribution des écarts, et le dernier tableau le nombre de spectres au-dessus de plusieurs seuils. La cellule prend environ 30 s. À regarder : le creux de l'histogramme autour de 1 Da.

In [ ]:
# Masse neutre : |z| × m/z = n × M + delta − z × m_e, d'où M = (|z| × m/z − delta + z × m_e) / n
p = train["adduct"].map(params)
n_mol, delta, z = (p.map(lambda t: t[i] if isinstance(t, tuple) else np.nan) for i in range(3))
train["neutral_mass"] = (z.abs() * train["precursor_mz"] - delta + z * ELECTRON_MASS) / n_mol

# Masse exacte de la vraie structure, calculée une fois par SMILES distinct (environ une minute)
smiles = train["normalized_smiles"].unique()
exact = {s: Descriptors.ExactMolWt(m) for s in smiles if (m := Chem.MolFromSmiles(s)) is not None}
train["mass_error_da"] = train["neutral_mass"] - train["normalized_smiles"].map(exact)

ppm = pd.read_parquet(TRAIN_PATH, columns=["precursor_error_ppm"])["precursor_error_ppm"]
ours_ppm = train["mass_error_da"] / train["neutral_mass"] * 1e6
display(pd.DataFrame({"precursor_error_ppm fourni": ppm, "notre écart (ppm)": ours_ppm})
        .groupby(train["adduct"]).median().loc[["[M+H]+", "[M-H]-", "[M+Na]+", "[2M+Na]+"]].round(2))

err = train["mass_error_da"].abs().clip(1e-5, 999)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(err.dropna(), bins=np.logspace(-5, 3, 120), color="#2a78d6")
ax.axvline(0.75, color="#52514e", linestyle="--", linewidth=1)
ax.text(0.8, ax.get_ylim()[1] * 0.9, "0,75 Da", color="#52514e")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("|masse déduite du précurseur − masse exacte du SMILES| (Da)"); ax.set_ylabel("spectres")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

thresholds = [0.01, 0.1, 0.5, 0.75, 1, 5, 50]
pd.DataFrame({"seuil (Da)": thresholds, "spectres au-dessus": [(err > t).sum() for t in thresholds],
              "% du train": [round((err > t).mean() * 100, 2) for t in thresholds]}).set_index("seuil (Da)")

### Les filtres `keep` et `query_ok`

Le seuil d'erreur de masse est un paramètre de `keep_mask`. On garde 0,75 Da par défaut, et on comparera plusieurs seuils (`CLEANING_THRESHOLDS`) sur le MRR@25 de la validation, à la section 9. Les requêtes de validation ne dépendent jamais de ce seuil. À regarder : la part de spectres gardés selon le seuil, et les bibliothèques les plus touchées.

In [ ]:
MASS_THRESHOLD = 0.75                     # Da, seuil retenu par défaut
CLEANING_THRESHOLDS = [None, 5, 1, 0.75, 0.1]  # seuils à comparer sur le MRR@25 en validation (None = pas de filtre)

def keep_mask(threshold=MASS_THRESHOLD):
    """Spectres fiables pour l'entraînement : aucune règle, et erreur de masse sous le seuil (None = pas de seuil)."""
    mass_ok = True if threshold is None else ~(train["mass_error_da"].abs() > threshold)
    return ~rules.any(axis=1) & mass_ok

# Requêtes de validation : seulement les règles qui n'utilisent pas la vraie structure
train["query_ok"] = ~rules.any(axis=1)
train["keep"] = keep_mask()

display(pd.DataFrame({"seuil (Da)": [str(t) for t in CLEANING_THRESHOLDS],
                      "spectres gardés pour l'entraînement (%)": [round(keep_mask(t).mean() * 100, 2) for t in CLEANING_THRESHOLDS]})
        .set_index("seuil (Da)"))
by_lib = pd.DataFrame({"écartés de l'entraînement (%)": (~train["keep"]).groupby(train["ingest_lib"]).mean() * 100,
                       "écartés des requêtes (%)": (~train["query_ok"]).groupby(train["ingest_lib"]).mean() * 100})
by_lib.round(2).sort_values("écartés de l'entraînement (%)", ascending=False)

### Décisions de nettoyage

| Règle | Ce qu'elle repère | Utilise la structure ? | Filtre |
|---|---|---|---|
| adduct inconnu | adduct illisible (`[Cat]2+`) : pas de masse neutre | non | `keep` et `query_ok` |
| charge contre mode | charge de l'adduct opposée au mode d'ionisation | non | `keep` et `query_ok` |
| adduct douteux | `[M-CH3]-` | non | `keep` et `query_ok` |
| erreur de masse | écart de plus de 0,75 Da entre la masse déduite et la masse exacte | oui | `keep` seulement |

Pourquoi 0,75 Da par défaut :
- une erreur d'adduct décale la masse d'au moins un atome d'hydrogène, soit 1,008 Da ; Na pris pour H fait 21,98 Da ;
- un précurseur noté à l'unité près, sur un appareil à basse résolution, s'écarte d'au plus 0,5 Da ;
- les appareils à haute résolution se trompent de quelques ppm, soit environ 0,001 Da pour une molécule de 300 Da : moins de 2 ppm avec un étalon interne et moins de 5 ppm avec un étalonnage externe sur Orbitrap (Makarov *et al.*, 2006), quelques ppm sur les appareils à temps de vol (Kind et Fiehn, 2006).

Un seuil en daltons, contrairement à un seuil en ppm, ne dépend pas de la taille de la molécule. Les autres seuils seront testés sur le MRR@25.

Sources :
- A. Makarov *et al.*, « Performance evaluation of a hybrid linear ion trap/orbitrap mass spectrometer », *Analytical Chemistry*, 78(7), 2113-2120, 2006 (PMID 16579588).
- T. Kind et O. Fiehn, « Metabolomic database annotations via query of elemental compositions: mass accuracy is insufficient even at less than 1 ppm », *BMC Bioinformatics*, 7, 234, 2006 (PMID 16646969).

### Le split par molécule

Pour l'instant, on ne travaille que sur le train. On le coupe selon les **molécules** (`inchikey14`), et non selon les spectres : tous les spectres d'une molécule vont du même côté. Un split par spectre mettrait des spectres d'une même molécule des deux côtés, et le modèle reconnaîtrait en validation des molécules déjà vues à l'entraînement. C'est une fuite, et le test caché ne la permet pas.

10 % des molécules vont en validation, et 2 000 d'entre elles forment le sous-ensemble `dev`, pour itérer vite. La graine est 42. L'entraînement utilise le filtre `keep`, les requêtes de validation le filtre `query_ok`. À regarder : aucune molécule en commun, et des bibliothèques qui pèsent autant des deux côtés.

In [ ]:
VAL_FRACTION = 0.10  # part des molécules mises en validation
N_DEV = 2_000        # molécules de validation gardées pour itérer vite

molecules = np.sort(train["inchikey14"].unique())  # trié : le tirage ne dépend pas de l'ordre du fichier
rng = np.random.default_rng(SEED)
val_molecules = rng.choice(molecules, size=round(len(molecules) * VAL_FRACTION), replace=False)
dev_molecules = rng.choice(np.sort(val_molecules), size=N_DEV, replace=False)

train["split"] = np.where(train["inchikey14"].isin(val_molecules), "val", "train")
train["dev"] = train["inchikey14"].isin(dev_molecules)

shared = set(train.loc[train["split"] == "train", "inchikey14"]) & set(train.loc[train["split"] == "val", "inchikey14"])
print(f"Molécules à la fois en entraînement et en validation : {len(shared)}")

groups = {"entraînement (keep)": (train["split"] == "train") & train["keep"],
          "validation (query_ok)": (train["split"] == "val") & train["query_ok"],
          "dont dev": train["dev"] & train["query_ok"]}
display(pd.DataFrame({name: {"spectres": m.sum(), "molécules": train.loc[m, "inchikey14"].nunique()}
                      for name, m in groups.items()}).T)

# Les bibliothèques doivent peser à peu près autant des deux côtés
(pd.crosstab(train["ingest_lib"], train["split"], normalize="columns") * 100).round(1).sort_values("train", ascending=False)

## 4. Catalogue et filtre à 10 ppm

**Catalogue.** Nous n'importons aucune bibliothèque de molécules externe. Le catalogue contient une structure par molécule du train et de la validation : c'est un « monde fermé », où la bonne réponse est toujours présente, même si le modèle n'a jamais vu ses spectres. Il est trié par masse exacte, pour retrouver vite les candidats d'une masse donnée. À regarder : le nombre de structures.

In [ ]:
from rdkit.Chem import rdFingerprintGenerator

# Catalogue « monde fermé » : une structure par molécule du train et de la validation.
# Aucune bibliothèque externe n'est importée ; le premier SMILES rencontré représente la molécule.
catalog = (train.drop_duplicates("inchikey14")[["inchikey14", "normalized_smiles"]]
           .assign(exact_mass=lambda d: d["normalized_smiles"].map(exact))
           .sort_values("exact_mass", ignore_index=True))
cat_keys = catalog["inchikey14"].to_numpy()
cat_masses = catalog["exact_mass"].to_numpy()
cat_row = pd.Series(np.arange(len(catalog)), index=cat_keys)  # inchikey14 -> ligne du catalogue
print(f"Catalogue : {len(catalog):,} structures, de {cat_masses.min():.1f} à {cat_masses.max():.1f} Da")

**Candidats.** Une requête correspond à une molécule `dev`. Sa masse neutre est la médiane des masses neutres de ses spectres (calculées à la section 3), ce qui protège contre un spectre isolé mal étiqueté. Ses candidats sont les structures du catalogue à ±10 ppm. À regarder : le rappel du filtre, c'est-à-dire la part des molécules dont la bonne structure est parmi les candidats. C'est le plafond de tout le pipeline.

In [ ]:
PPM = 10  # tolérance du filtre par masse

# Une requête par molécule dev : ses spectres query_ok, et la médiane de leurs masses neutres
queries = (train[train["dev"] & train["query_ok"]].groupby("inchikey14")
           .agg(neutral_mass=("neutral_mass", "median")).reset_index())

def candidates(mass, ppm=PPM):
    """Lignes du catalogue dont la masse exacte est à ± ppm de mass (le catalogue est trié par masse)."""
    lo, hi = np.searchsorted(cat_masses, [mass * (1 - ppm * 1e-6), mass * (1 + ppm * 1e-6)])
    return np.arange(lo, hi)

queries["cands"] = queries["neutral_mass"].map(candidates)
n_cands = queries["cands"].map(len)
recall = np.mean([cat_row[k] in c for k, c in zip(queries["inchikey14"], queries["cands"])])
print(f"{len(queries):,} molécules dev ; candidats par molécule : médiane {n_cands.median():.0f}, "
      f"90e centile {n_cands.quantile(0.9):.0f} ; rappel du filtre : {recall:.1%}")

## 5. MRR@25 et baseline 0

`mrr_at_k` calcule la métrique de la compétition, et une ligne la vérifie sur un petit exemple. `evaluate` classe les candidats de chaque requête avec une fonction de score, puis calcule le MRR@25. Toutes les méthodes passent par elle, sur les mêmes requêtes.

In [ ]:
def mrr_at_k(true_keys, ranked_keys, k=25):
    """MRR@k : moyenne de 1/rang de la bonne réponse parmi les k premières propositions (0 si absente)."""
    scores = []
    for truth, ranked in zip(true_keys, ranked_keys):
        top = list(ranked[:k])
        scores.append(1 / (top.index(truth) + 1) if truth in top else 0.0)
    return float(np.mean(scores))

assert mrr_at_k(["a", "b", "c"], [["a", "x"], ["x", "b"], ["x", "y"]]) == (1 + 0.5 + 0) / 3

def evaluate(score_fn, k=25):
    """Classe les candidats de chaque requête par score décroissant, puis calcule le MRR@k.

    score_fn(i, cands) renvoie un score par candidat de la requête i (plus grand = plus probable).
    """
    ranked = []
    for i, cands in enumerate(queries["cands"]):
        order = np.argsort(-score_fn(i, cands), kind="stable")
        ranked.append(cat_keys[cands[order]])
    return mrr_at_k(queries["inchikey14"], ranked, k)

La baseline 0 classe les candidats par écart de masse, sans regarder le spectre. C'est le plancher : un modèle doit faire mieux pour être utile.

In [ ]:
# Baseline 0 : le candidat le plus proche en masse est classé premier
baseline_mrr = evaluate(lambda i, c: -np.abs(cat_masses[c] - queries["neutral_mass"][i]))
print(f"Baseline 0 (tri par erreur de masse) : MRR@25 = {baseline_mrr:.3f}")

## 6. Fingerprints de Morgan et oracle

On calcule le fingerprint de Morgan (rayon 2, 2048 bits) de chaque structure du catalogue, en une minute environ. Ils sont stockés en bits compactés pour tenir en mémoire. Un modèle prédit un fingerprint, puis les candidats sont classés par similarité de Tanimoto. Comme les prédictions sont des probabilités, on utilise le Tanimoto continu, Σpc / (Σp + Σc − Σpc). L'oracle classe avec le vrai fingerprint : c'est le plafond de cette représentation. À regarder : l'écart entre l'oracle et la baseline 0.

In [ ]:
FP_RADIUS, FP_SIZE = 2, 2048  # Morgan rayon 2 (équivalent ECFP4), 2048 bits
morgan = rdFingerprintGenerator.GetMorganGenerator(radius=FP_RADIUS, fpSize=FP_SIZE)

# Fingerprint de chaque structure du catalogue (environ 1 min), stocké en bits compactés : 256 octets par molécule
cat_fps = np.packbits(np.stack([morgan.GetFingerprintAsNumPy(Chem.MolFromSmiles(s))
                                for s in catalog["normalized_smiles"]]), axis=1)

def fingerprints(rows):
    """Fingerprints (0/1, float32) des lignes rows du catalogue."""
    return np.unpackbits(cat_fps[rows], axis=1).astype(np.float32)

def tanimoto(pred, cand_fps):
    """Tanimoto continu entre un fingerprint prédit (probabilités) et des fingerprints binaires."""
    inter = cand_fps @ pred
    return inter / (pred.sum() + cand_fps.sum(axis=1) - inter + 1e-9)

# Oracle : on classe avec le vrai fingerprint, le mieux que permet cette représentation
true_fps = fingerprints(cat_row[queries["inchikey14"]].to_numpy())
oracle_mrr = evaluate(lambda i, c: tanimoto(true_fps[i], fingerprints(c)))
print(f"Oracle (vrai fingerprint) : MRR@25 = {oracle_mrr:.3f}")

## 7. Binning des spectres

**Lecture des pics.** Les pics sont trop lourds pour tout le train. On lit ceux de 200 000 spectres d'entraînement tirés au hasard, et ceux des spectres `dev`. L'échantillon est tiré dans tout le split d'entraînement, avant nettoyage, pour pouvoir comparer ensuite plusieurs seuils de masse.

In [ ]:
import scipy.sparse as sp
from sklearn.preprocessing import normalize

N_TRAIN_SAMPLE = 200_000  # spectres d'entraînement tirés au hasard (tout le train serait trop long sur CPU)

def read_peaks(row_ids):
    """Pics des spectres demandés (positions dans train.parquet), lus bloc par bloc pour limiter la mémoire."""
    pf = pq.ParquetFile(TRAIN_PATH)
    starts = np.cumsum([0] + [pf.metadata.row_group(g).num_rows for g in range(pf.num_row_groups)])
    row_ids = np.sort(row_ids)
    parts = []
    for g in range(pf.num_row_groups):
        local = row_ids[(row_ids >= starts[g]) & (row_ids < starts[g + 1])] - starts[g]
        if len(local):
            parts.append(pf.read_row_group(g, columns=["ms2_mzs", "ms2_normalized_intensities"]).take(local).to_pandas())
    return pd.concat(parts, ignore_index=True).set_index(row_ids)

rng = np.random.default_rng(SEED)
train_ids = np.sort(rng.choice(np.flatnonzero(train["split"] == "train"), N_TRAIN_SAMPLE, replace=False))
dev_ids = np.flatnonzero(train["dev"] & train["query_ok"])
peaks = read_peaks(np.concatenate([train_ids, dev_ids]))
print(f"Pics lus : {len(train_ids):,} spectres d'entraînement, {len(dev_ids):,} spectres dev")

**Binning.** On découpe l'axe m/z en cases régulières jusqu'à 1 000 m/z. Chaque case reçoit le pic le plus haut qui y tombe, après une transformation des intensités (racine carrée par défaut), puis chaque spectre est normalisé (norme 1). Le produit scalaire de deux spectres est alors leur similarité cosinus. La largeur des cases et la transformation sont des hyperparamètres, testés à la section 8. À regarder : le nombre de cases non nulles par spectre.

In [ ]:
TRANSFORMS = {"aucune": lambda x: x, "racine": np.sqrt, "log": lambda x: np.log1p(100 * x) / np.log1p(100)}

def bin_spectra(peaks, bin_width=0.1, max_mz=1000, transform="racine"):
    """Matrice creuse (spectres × cases) : le pic le plus haut de chaque case, puis lignes de norme 1."""
    lengths = peaks["ms2_mzs"].map(len).to_numpy()
    df = pd.DataFrame({"row": np.repeat(np.arange(len(peaks)), lengths),
                       "mz": np.concatenate(peaks["ms2_mzs"].to_numpy()),
                       "it": np.concatenate(peaks["ms2_normalized_intensities"].to_numpy())})
    df = df[(df["mz"] >= 0) & (df["mz"] < max_mz) & (df["it"] > 0)]
    df["col"] = np.floor(np.round(df["mz"] / bin_width, 6)).astype(int)  # l'arrondi évite 0.3 / 0.1 = 2.999…
    cells = df.groupby(["row", "col"])["it"].max().reset_index()
    X = sp.csr_matrix((TRANSFORMS[transform](cells["it"].to_numpy()), (cells["row"], cells["col"])),
                      shape=(len(peaks), int(round(max_mz / bin_width))), dtype=np.float32)
    return normalize(X)

X_example = bin_spectra(peaks.loc[dev_ids[:1000]])
print(f"{X_example.shape[1]:,} cases ; {X_example.nnz / X_example.shape[0]:.0f} cases non nulles par spectre")

## 8. Modèles

**Imputation.** Le MLP utilise l'énergie de collision. Ses valeurs manquantes, après la formule NCE → eV, sont remplacées par la médiane de leur famille d'instrument et de leur mode d'ionisation. Les médianes sont apprises sur l'entraînement seulement. La variable `ce_manquante` vaut 1 pour une énergie imputée. À regarder : le tableau des médianes.

In [ ]:
# Médianes apprises sur l'entraînement seulement, puis appliquées à tous les spectres
train["instrument_family"] = instrument_family(train["instrument_type"])
is_fit = (train["split"] == "train") & train["keep"]
group_median = train[is_fit].groupby(["instrument_family", "ionization_mode"])["ce_mean"].median().rename("median")
fallback = train.loc[is_fit, "ce_mean"].median()  # groupe sans aucune énergie connue

by_group = train[["instrument_family", "ionization_mode"]].join(group_median, on=["instrument_family", "ionization_mode"])
train["ce_manquante"] = train["ce_mean"].isna()
train["ce_filled"] = train["ce_mean"].fillna(by_group["median"]).fillna(fallback)
print(f"Énergies imputées : {train['ce_manquante'].sum():,} ; restantes : {train['ce_filled'].isna().sum()}")
group_median.unstack()

**Métadonnées.** Pour le MLP, chaque spectre reçoit aussi : le mode d'ionisation, la masse neutre / 1 000, le log10 du nombre de pics, l'énergie / 100, `ce_manquante`, l'adduct en one-hot (les 15 plus fréquents de l'entraînement, plus « autre ») et la famille d'instrument en one-hot. Les colonnes interdites (structure, `inchikey14`, `precursor_error_ppm`) n'y figurent pas.

In [ ]:
TOP_ADDUCTS = list(train.loc[is_fit, "adduct"].value_counts().index[:15])  # appris sur l'entraînement
FAMILIES = [name for name, _ in INSTRUMENT_FAMILIES] + ["Autre", "Inconnu"]

def meta_features(df):
    """Métadonnées d'un spectre : nombres mis à l'échelle, puis adduct et instrument en one-hot."""
    numeric = pd.DataFrame({
        "mode_positif": df["ionization_mode"] == "positive",
        "masse_neutre": df["neutral_mass"] / 1000,
        "log_num_peaks": np.log10(df["num_peaks"]),
        "energie": df["ce_filled"] / 100,
        "ce_manquante": df["ce_manquante"],
    })
    adduct = pd.Categorical(df["adduct"].where(df["adduct"].isin(TOP_ADDUCTS), "autre"), categories=TOP_ADDUCTS + ["autre"])
    instrument = pd.Categorical(df["instrument_family"], categories=FAMILIES)
    onehot = [pd.get_dummies(adduct, prefix="adduct"), pd.get_dummies(instrument, prefix="instrument")]
    return pd.concat([numeric, *[o.set_index(df.index) for o in onehot]], axis=1).astype(np.float32)

meta_features(train.loc[dev_ids[:3]])

**kNN.** Il n'y a rien à apprendre. Pour un spectre `dev`, on cherche les k spectres d'entraînement les plus proches (similarité cosinus des spectres binés). Le fingerprint prédit est la moyenne de leurs fingerprints, pondérée par la similarité. On fait la moyenne sur les spectres d'une molécule. La recherche se fait par blocs de 200 requêtes, pour limiter la mémoire.

In [ ]:
def molecule_fps(spectrum_fps, row_ids):
    """Moyenne des fingerprints prédits des spectres de chaque requête, dans l'ordre de queries."""
    keys = train["inchikey14"].to_numpy()[row_ids]
    return pd.DataFrame(spectrum_fps).groupby(keys).mean().loc[queries["inchikey14"]].to_numpy(dtype=np.float32)

def reference_ids(threshold):
    """Spectres de l'échantillon d'entraînement gardés par keep_mask(threshold)."""
    return train_ids[keep_mask(threshold).to_numpy()[train_ids]]

def train_knn(bin_width=0.1, transform="racine", k=20, threshold=MASS_THRESHOLD):
    """Plus proches voisins : rien à apprendre. La base binée est construite à la prédiction, pour la mémoire."""
    ref_ids = reference_ids(threshold)
    ref_rows = cat_row[train["inchikey14"].to_numpy()[ref_ids]].to_numpy()  # ligne du catalogue de chaque voisin

    def predict(query_ids, block=200):
        """Fingerprint prédit = moyenne des fingerprints des k voisins, pondérée par la similarité cosinus."""
        X_ref = bin_spectra(peaks.loc[ref_ids], bin_width, transform=transform)
        X_q = bin_spectra(peaks.loc[query_ids], bin_width, transform=transform)
        preds = []
        for start in range(0, X_q.shape[0], block):
            sim = (X_q[start:start + block] @ X_ref.T).toarray()
            top = np.argpartition(-sim, k, axis=1)[:, :k]
            w = np.take_along_axis(sim, top, axis=1)
            neighbours = fingerprints(ref_rows[top].ravel()).reshape(len(top), k, FP_SIZE)
            preds.append(np.einsum("qk,qkb->qb", w, neighbours) / (w.sum(axis=1, keepdims=True) + 1e-9))
        return np.vstack(preds)
    return predict

**MLP.** Un réseau à une couche cachée de 1 024 neurones (ReLU, dropout 0,2) prédit les 2 048 bits à partir du spectre biné et, en option, des métadonnées. La perte est la BCE, une classification oui/non par bit, avec Adam (taux 10⁻³), des batchs de 512 et 8 époques. Le réseau est volontairement petit, pour tourner sur CPU.

In [ ]:
import torch
from torch import nn

def train_mlp(use_meta=True, hidden=1024, epochs=8, lr=1e-3, batch_size=512, threshold=MASS_THRESHOLD):
    """MLP : spectre biné (et métadonnées) -> 2048 probabilités, une par bit, appris avec la BCE."""
    torch.manual_seed(SEED)
    def inputs(ids):
        X = bin_spectra(peaks.loc[ids])
        return sp.hstack([X, sp.csr_matrix(meta_features(train.loc[ids]).to_numpy())], format="csr") if use_meta else X

    ref_ids = reference_ids(threshold)
    X, rows = inputs(ref_ids), cat_row[train["inchikey14"].to_numpy()[ref_ids]].to_numpy()
    model = nn.Sequential(nn.Linear(X.shape[1], hidden), nn.ReLU(), nn.Dropout(0.2), nn.Linear(hidden, FP_SIZE))
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()
    rng = np.random.default_rng(SEED)
    for epoch in range(epochs):
        model.train()
        order, total = rng.permutation(X.shape[0]), 0.0
        for start in range(0, len(order), batch_size):
            batch = order[start:start + batch_size]
            loss = loss_fn(model(torch.from_numpy(X[batch].toarray())), torch.from_numpy(fingerprints(rows[batch])))
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            total += loss.item() * len(batch)
        print(f"  époque {epoch + 1} : loss {total / len(order):.4f}")

    def predict(query_ids):
        model.eval()
        X_q = inputs(query_ids)
        with torch.no_grad():
            return np.vstack([torch.sigmoid(model(torch.from_numpy(X_q[s:s + 4096].toarray()))).numpy()
                              for s in range(0, X_q.shape[0], 4096)])
    return predict

### Entraînements

Les configurations kNN changent un seul hyperparamètre à la fois par rapport à la référence : la largeur des cases, la transformation des intensités, le nombre de voisins k et le seuil du filtre de masse (`CLEANING_THRESHOLDS`, section 3). Le seuil ne change que la base d'entraînement, jamais les requêtes. Cette cellule est rapide : un kNN n'a rien à apprendre.

In [ ]:
# On change un seul hyperparamètre à la fois par rapport à la configuration de référence
KNN_CONFIGS = {
    "kNN référence (0,1 Da, racine, k = 20, seuil 0,75 Da)": {},
    "kNN cases de 0,01 Da": {"bin_width": 0.01},
    "kNN cases de 1 Da": {"bin_width": 1},
    "kNN intensités brutes": {"transform": "aucune"},
    "kNN intensités en log": {"transform": "log"},
    "kNN k = 5": {"k": 5},
    "kNN k = 50": {"k": 50},
    "kNN sans filtre de masse": {"threshold": None},
    "kNN seuil de masse 0,1 Da": {"threshold": 0.1},
}
models = {name: train_knn(**config) for name, config in KNN_CONFIGS.items()}
print(f"{len(models)} configurations kNN prêtes")

On entraîne deux MLP, avec et sans métadonnées, pour mesurer ce qu'elles apportent. Chacun prend environ 12 minutes sur CPU. À regarder : la loss doit baisser à chaque époque.

In [ ]:
print("MLP spectre + métadonnées")
models["MLP spectre + métadonnées"] = train_mlp(use_meta=True)
print("MLP spectre seul")
models["MLP spectre seul"] = train_mlp(use_meta=False)

## 9. Résultats sur la validation

Chaque modèle prédit un fingerprint pour chaque molécule `dev`, puis `evaluate` mesure le MRR@25 sur les mêmes requêtes. La baseline 0 et l'oracle encadrent les résultats. La cellule prend environ 25 minutes, surtout pour les kNN (2 minutes chacun). À regarder : quels hyperparamètres font gagner le kNN, et si le MLP dépasse le kNN.

In [ ]:
results = {"Baseline 0 (tri par masse)": baseline_mrr, "Oracle (vrai fingerprint)": oracle_mrr}
for name, predict in models.items():
    pred = molecule_fps(predict(dev_ids), dev_ids)  # un fingerprint prédit par molécule dev
    results[name] = evaluate(lambda i, c: tanimoto(pred[i], fingerprints(c)))
    print(f"{name:55s} MRR@25 = {results[name]:.3f}")

pd.Series(results, name="MRR@25 (dev)").sort_values(ascending=False).round(3).to_frame()